**# 1 — Project setup & dependencies**

In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


# ---------------------------------------------------------
# Repository configuration
# ---------------------------------------------------------

REPO_URL = (
    "https://github.com/natsumeSama/"
    "DEAP-EEG-video-fusion-experiment-pipeline.git"
)

REPO_NAME = "DEAP-EEG-video-fusion-experiment-pipeline"


# ---------------------------------------------------------
# Detect Google Colab
# ---------------------------------------------------------

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------

cwd = Path.cwd()


# Case 1: notebook launched locally from /notebooks
if cwd.name == "notebooks" and (cwd.parent / "src").exists():

    PROJECT_ROOT = cwd.parent


# Case 2: already running from the repository root
elif (cwd / "src" / "deap_fusion").exists():

    PROJECT_ROOT = cwd


# Case 3: fresh Google Colab runtime
elif IN_COLAB:

    PROJECT_ROOT = Path("/content") / REPO_NAME

    if not PROJECT_ROOT.exists():

        print("Cloning GitHub repository...")

        subprocess.run(
            [
                "git",
                "clone",
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    else:

        print(
            "Repository already exists in this Colab runtime."
        )


# Anything else means the repository could not be located
else:

    raise FileNotFoundError(
        "Could not locate the project repository. "
        "Run the notebook from the repository root "
        "or from the notebooks/ directory."
    )


# ---------------------------------------------------------
# Add src/ to Python path
# ---------------------------------------------------------

SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))


# Work from repository root
os.chdir(PROJECT_ROOT)


# ---------------------------------------------------------
# Dependency setup
# ---------------------------------------------------------

# module name -> pip package name
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "cv2": "opencv-python",
    "tqdm": "tqdm",
    "gdown": "gdown",
}


missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]


if missing_packages:

    print(
        "Installing missing packages:",
        ", ".join(missing_packages),
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
    )

    print("Dependency installation complete.")

else:

    print("All required dependencies are already installed.")


# ---------------------------------------------------------
# Environment summary
# ---------------------------------------------------------

print()
print(
    "Environment:",
    "Google Colab" if IN_COLAB else "Local",
)
print("Project root:", PROJECT_ROOT)
print("Source root:", SRC_ROOT)

Cloning GitHub repository...
All required dependencies are already installed.

Environment: Google Colab
Project root: /content/DEAP-EEG-video-fusion-experiment-pipeline
Source root: /content/DEAP-EEG-video-fusion-experiment-pipeline/src


**# 2. IMPORTS**

In [2]:
import pandas as pd


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

from deap_fusion.config import (
    LABEL_TYPE,
    MAX_SUBJECT_ID,
    NUM_FRAMES,
    BATCH_SIZE,
    WEIGHT_DECAY,
    EPOCHS,
    LR,
    FREEZE_BACKBONE,
    TRAIN_RATIO,
    SEED,
    NUM_WORKERS,
)


# ---------------------------------------------------------
# Data
# ---------------------------------------------------------

from deap_fusion.data.download import (
    data_download,
)

from deap_fusion.data.splits import (
    init_shared_splits,
    get_shared_split,
)


# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

from deap_fusion.training.common import (
    set_seeds,
)


# ---------------------------------------------------------
# Experiments
# ---------------------------------------------------------

from deap_fusion.experiments.main import (
    run_main_experiment,
)

from deap_fusion.experiments.cv import (
    run_Kfold_cv_all_models,
)


# ---------------------------------------------------------
# Evaluation results
# ---------------------------------------------------------

from deap_fusion.evaluation.results import (
    MODEL_DISPLAY_NAMES,

    # Main experiment
    standardize_eeg_results,
    standardize_video_results,
    standardize_fusion_results,
    build_subject_metrics_table,
    build_summary_table,
    collect_main_film_stats,
    build_main_film_summary_table,
    save_tables_to_csv,

    # Cross-validation
    collect_cv_results,
    build_cv_fold_table,
    summarize_cv_results,
    make_cv_paper_table,
    compare_cv_models,
    compare_two_cv_models,
    compare_all_metrics,
    compare_film_vs_concat_all_metrics,
    collect_cv_film_stats,
    build_cv_film_summary_table,
)


# ---------------------------------------------------------
# Evaluation plots
# ---------------------------------------------------------

from deap_fusion.evaluation.plots import (
    # Main experiment
    plot_metric_bar,
    plot_mean_curves,
    plot_metric_box,

    # Cross-validation
    plot_cv_summary_bar,
    plot_cv_fold_lines,
    plot_subject_heatmap,
    plot_cv_model_difference,

    # FiLM analysis
    plot_film_parameter_histograms,
    plot_film_boxplot_by_label,
    plot_film_boxplot_correct_vs_wrong,
)


print("Project imports loaded.")

Project imports loaded.


**# 3. USER CONFIGURATION**

In [ ]:
# ---------------------------------------------------------
# Dataset source
# ---------------------------------------------------------

# Options:
#
# "local"
#     Use datasets already placed under ./data/
#
# "drive_download"
#     Download dataset ZIP files from Google Drive using
#     the file IDs below.

DATA_SOURCE = ""


# ---------------------------------------------------------
# Google Drive file IDs
# Used only when DATA_SOURCE = "drive_download"
# ---------------------------------------------------------

DEAP_EEG_DRIVE_ID = ""
DEAP_FACE_CROPS_DRIVE_ID = ""
DEAP_VIDEO_DRIVE_ID = ""


# Raw videos are not required for normal experiments.
# Precomputed face crops can be used directly.
DOWNLOAD_RAW_VIDEOS = False


# ---------------------------------------------------------
# Experiment configuration
# Imported from src/deap_fusion/config.py
# ---------------------------------------------------------

TARGET_NAME = (
    "valence"
    if LABEL_TYPE == 0
    else "arousal"
)


print("=" * 60)
print("USER CONFIGURATION")
print("=" * 60)

print()
print("Dataset")
print("-" * 60)
print(f"Data source:           {DATA_SOURCE}")
print(f"Download raw videos:   {DOWNLOAD_RAW_VIDEOS}")

print()
print("Experiment")
print("-" * 60)
print(f"Target:                {TARGET_NAME}")
print(f"Label type:            {LABEL_TYPE}")
print(f"Max subject id:        {MAX_SUBJECT_ID}")
print(f"Video frames/trial:    {NUM_FRAMES}")
print(f"Batch size:            {BATCH_SIZE}")
print(f"Epochs:                {EPOCHS}")
print(f"Learning rate:         {LR}")
print(f"Weight decay:          {WEIGHT_DECAY}")
print(f"Train ratio:           {TRAIN_RATIO}")
print(f"Seed:                  {SEED}")
print(f"Workers:               {NUM_WORKERS}")
print(f"Freeze video backbone: {FREEZE_BACKBONE}")

**# 4. DATASET SETUP & VALIDATION**

In [ ]:
# ---------------------------------------------------------
# Resolve dataset paths
# ---------------------------------------------------------

if DATA_SOURCE == "local":

    DEAP_ROOT = PROJECT_ROOT / "data" / "eeg"
    IMAGE_ROOT = PROJECT_ROOT / "data" / "face_crops"
    VIDEO_ROOT = PROJECT_ROOT / "data" / "videos"


elif DATA_SOURCE == "drive_download":

    if not DEAP_EEG_DRIVE_ID:
        raise ValueError(
            "DEAP_EEG_DRIVE_ID is empty. "
            "Provide a Google Drive file ID or use DATA_SOURCE='local'."
        )

    if not DEAP_FACE_CROPS_DRIVE_ID:
        raise ValueError(
            "DEAP_FACE_CROPS_DRIVE_ID is empty. "
            "Provide a Google Drive file ID or use DATA_SOURCE='local'."
        )

    DEAP_ROOT = data_download(
        DEAP_EEG_DRIVE_ID,
        "eeg",
    )

    IMAGE_ROOT = data_download(
        DEAP_FACE_CROPS_DRIVE_ID,
        "face_crops",
    )

    VIDEO_ROOT = PROJECT_ROOT / "data" / "videos"

    if DOWNLOAD_RAW_VIDEOS:

        if not DEAP_VIDEO_DRIVE_ID:
            raise ValueError(
                "DEAP_VIDEO_DRIVE_ID is empty."
            )

        VIDEO_ROOT = data_download(
            DEAP_VIDEO_DRIVE_ID,
            "videos",
        )


else:

    raise ValueError(
        "DATA_SOURCE must be 'local' or 'drive_download'."
    )


# ---------------------------------------------------------
# Dataset paths
# ---------------------------------------------------------

print("EEG root:       ", DEAP_ROOT)
print("Video root:     ", VIDEO_ROOT)
print("Face crops root:", IMAGE_ROOT)


# ---------------------------------------------------------
# EEG sanity check
# ---------------------------------------------------------

eeg_dat_dir = (
    DEAP_ROOT
    / "data_preprocessed_python"
)

if not eeg_dat_dir.exists():
    raise FileNotFoundError(
        f"EEG folder not found: {eeg_dat_dir}"
    )

eeg_files = sorted(
    eeg_dat_dir.glob("*.dat")
)

if len(eeg_files) == 0:
    raise RuntimeError(
        f"No DEAP .dat files found under {eeg_dat_dir}"
    )


# ---------------------------------------------------------
# Face crop sanity check
# ---------------------------------------------------------

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Face crops folder not found: {IMAGE_ROOT}"
    )

face_trial_dirs = [
    p
    for p in IMAGE_ROOT.iterdir()
    if p.is_dir()
]


# ---------------------------------------------------------
# Dataset summary
# ---------------------------------------------------------

print()
print(f"EEG subjects found: {len(eeg_files)}")
print(
    f"Face trial folders found: "
    f"{len(face_trial_dirs)}"
)

print("Dataset setup OK.")

**# 5. REPRODUCIBILITY & SHARED SPLIT CHECK**

In [ ]:
# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

set_seeds(SEED)

print(f"Random seed initialized to {SEED}")


# ---------------------------------------------------------
# Initialize shared subject-dependent splits
# ---------------------------------------------------------

init_shared_splits(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,
    label_type=LABEL_TYPE,
    max_subject_id=MAX_SUBJECT_ID,
    split_mode="subject_dependent",
    train_ratio=TRAIN_RATIO,
    seed=SEED,
)


# ---------------------------------------------------------
# Inspect one example subject
# ---------------------------------------------------------

example_sid = "s01"

train_pairs, val_pairs, split_info = get_shared_split(
    sid=example_sid,
    split_mode="subject_dependent",
)


print()
print("=" * 60)
print("SHARED SPLIT SANITY CHECK")
print("=" * 60)

print(f"Subject: {example_sid}")

print()
print("Split info:")
print(split_info)

print()
print("Train trials:", len(train_pairs))
print("Validation trials:", len(val_pairs))

print()
print(
    "First train pairs:",
    train_pairs[:5],
)

print(
    "First validation pairs:",
    val_pairs[:5],
)

**# 6. MAIN EXPERIMENT - SINGLE 75/25 SHARED SPLIT**

In [ ]:
# ---------------------------------------------------------
# Run all models on the same subject-dependent split
# ---------------------------------------------------------

main_results = run_main_experiment(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,
    label_type=LABEL_TYPE,
    seed=SEED,
)


# ---------------------------------------------------------
# Extract model results
# ---------------------------------------------------------

all_eeg_sd = main_results["EEG"]
all_video_sd = main_results["VIDEO"]
all_fusion_sd = main_results["FUSION"]
all_concat_fusion_sd = main_results["CONCAT"]
all_decision_fusion_sd = main_results["DECISION"]


# ---------------------------------------------------------
# Quick check
# ---------------------------------------------------------

print()
print("=" * 60)
print("MAIN EXPERIMENT QUICK CHECK")
print("=" * 60)

for model_name, result in main_results.items():

    if result is None:
        print(f"{model_name}: missing")
        continue

    subject_results = result.get(
        "subject_results",
        {},
    )

    print(
        f"{model_name}: "
        f"{len(subject_results)} subject results"
    )

**# 7. MAIN EXPERIMENT - ANALYSIS & RESULTS**

In [ ]:
# ---------------------------------------------------------
# Standardize model results
# ---------------------------------------------------------

eeg_std = standardize_eeg_results(
    all_eeg_sd,
    model_name="EEG",
)

video_std = standardize_video_results(
    all_video_sd,
    model_name="VIDEO",
)

film_std = standardize_fusion_results(
    all_fusion_sd,
    model_name="FUSION",
)

concat_std = standardize_fusion_results(
    all_concat_fusion_sd,
    model_name="CONCAT",
)

decision_std = standardize_fusion_results(
    all_decision_fusion_sd,
    model_name="DECISION",
)


main_std_results = {
    "EEG": eeg_std,
    "VIDEO": video_std,
    "FUSION": film_std,
    "CONCAT": concat_std,
    "DECISION": decision_std,
}


print("Main experiment results standardized.")


# ---------------------------------------------------------
# Subject-level and aggregate result tables
# ---------------------------------------------------------

main_subject_tables = {}
main_summary_tables = []


for model_name, std_results in main_std_results.items():

    # Per-subject metrics
    subject_df = build_subject_metrics_table(
        std_results
    ).copy()

    subject_df.insert(
        0,
        "model",
        model_name,
    )

    main_subject_tables[model_name] = subject_df


    # Aggregate metrics across subjects
    summary_df = build_summary_table(
        std_results
    )

    main_summary_tables.append(
        summary_df
    )


# Combine all models
main_subject_df = pd.concat(
    main_subject_tables.values(),
    ignore_index=True,
)

main_summary_df = pd.concat(
    main_summary_tables,
    ignore_index=True,
)


print()
print("Main 75/25 split - subject-level results")
display(main_subject_df)

print()
print("Main 75/25 split - model summary")
display(main_summary_df)


# ---------------------------------------------------------
# Paper-ready comparison table
# ---------------------------------------------------------

def format_mean_std(row, metric):

    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    mean_value = row.get(mean_col)
    std_value = row.get(std_col)

    if pd.isna(mean_value):
        return ""

    if pd.isna(std_value):
        std_value = 0.0

    return (
        f"{mean_value * 100:.2f} ± "
        f"{std_value * 100:.2f}"
    )


main_paper_rows = []


for _, row in main_summary_df.iterrows():

    model_name = row["model_name"]

    main_paper_rows.append({
        "Model": MODEL_DISPLAY_NAMES.get(
            model_name,
            model_name,
        ),
        "Best Val Acc": format_mean_std(
            row,
            "best_val_acc",
        ),
        "Accuracy": format_mean_std(
            row,
            "acc",
        ),
        "Balanced Acc": format_mean_std(
            row,
            "balanced_acc",
        ),
        "Precision": format_mean_std(
            row,
            "precision",
        ),
        "Recall": format_mean_std(
            row,
            "recall",
        ),
        "F1": format_mean_std(
            row,
            "f1",
        ),
    })


main_paper_table = pd.DataFrame(
    main_paper_rows
)


print()
print("Main 75/25 split - paper-ready comparison")
display(main_paper_table)


# ---------------------------------------------------------
# FiLM modulation analysis
# ---------------------------------------------------------

main_film_stats_df = collect_main_film_stats(
    all_fusion_sd
)

main_film_summary_df = build_main_film_summary_table(
    main_film_stats_df
)


print()
print(
    "Main 75/25 split - "
    "FiLM trial-level modulation statistics"
)
display(main_film_stats_df)

print()
print(
    "Main 75/25 split - "
    "FiLM modulation summary"
)
display(main_film_summary_df)


# ---------------------------------------------------------
# Main experiment performance plots
# ---------------------------------------------------------

# Per-subject validation accuracy
for model_name, std_results in main_std_results.items():

    plot_metric_bar(
        std_results,
        metric="best_val_acc",
        sort=False,
    )


# Mean training curves
# Decision fusion is excluded because it has no training.
for model_name in [
    "EEG",
    "VIDEO",
    "FUSION",
    "CONCAT",
]:

    plot_mean_curves(
        main_std_results[model_name]
    )


# Metric distributions across subjects
for model_name, std_results in main_std_results.items():

    plot_metric_box(
        std_results,
        metrics=[
            "best_val_acc",
            "balanced_acc",
            "precision",
            "recall",
            "f1",
        ],
    )


# ---------------------------------------------------------
# FiLM modulation plots
# ---------------------------------------------------------

# Overall FiLM parameter distributions
plot_film_parameter_histograms(
    main_film_stats_df
)


# FiLM modulation according to emotion class
plot_film_boxplot_by_label(
    main_film_stats_df,
    "gamma_deviation_from_1",
)

plot_film_boxplot_by_label(
    main_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_by_label(
    main_film_stats_df,
    "gate_mean",
)

plot_film_boxplot_by_label(
    main_film_stats_df,
    "relative_modulation",
)


# FiLM modulation: correct vs incorrect predictions
plot_film_boxplot_correct_vs_wrong(
    main_film_stats_df,
    "relative_modulation",
)

plot_film_boxplot_correct_vs_wrong(
    main_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_correct_vs_wrong(
    main_film_stats_df,
    "gate_mean",
)


# ---------------------------------------------------------
# Save main experiment results
# ---------------------------------------------------------

main_out_dir = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "main"
)

main_out_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# Combined subject-level results
main_subject_df.to_csv(
    main_out_dir
    / f"main_subject_results_label{LABEL_TYPE}.csv",
    index=False,
)


# Combined model summaries
main_summary_df.to_csv(
    main_out_dir
    / f"main_summary_label{LABEL_TYPE}.csv",
    index=False,
)


# Paper-ready comparison table
main_paper_table.to_csv(
    main_out_dir
    / f"main_paper_table_label{LABEL_TYPE}.csv",
    index=False,
)


# FiLM modulation analysis
main_film_stats_df.to_csv(
    main_out_dir
    / f"main_film_stats_label{LABEL_TYPE}.csv",
    index=False,
)

main_film_summary_df.to_csv(
    main_out_dir
    / f"main_film_summary_label{LABEL_TYPE}.csv",
    index=False,
)


# Individual model tables
for model_name, std_results in main_std_results.items():

    save_tables_to_csv(
        std_results,
        out_dir=main_out_dir,
        prefix=(
            f"{model_name.lower()}"
            f"_label{LABEL_TYPE}"
        ),
    )


print()
print(
    "Saved main experiment results to:",
    main_out_dir,
)

**# 8. 4-FOLD CROSS-VALIDATION**

In [ ]:
# ---------------------------------------------------------
# Run all models across shared CV folds
# ---------------------------------------------------------

cv_results = run_Kfold_cv_all_models(
    eeg_root=DEAP_ROOT,
    image_root=IMAGE_ROOT,

    label_type=LABEL_TYPE,
    split_mode="subject_dependent",
    n_splits=4,
    max_subject_id=MAX_SUBJECT_ID,
    seed=SEED,

    run_eeg=True,
    run_video=True,
    run_film=True,
    run_concat=True,
    run_decision=True,

    eeg_mode=2,
    lambda_cons=0.0,

    eeg_epochs=EPOCHS,
    video_epochs=EPOCHS,

    fusion_stage1_epochs=15,
    fusion_stage2_epochs=5,

    batch_size=BATCH_SIZE,
)


# ---------------------------------------------------------
# Quick check
# ---------------------------------------------------------

print()
print("=" * 60)
print("CV QUICK CHECK")
print("=" * 60)

print(
    "Number of completed folds:",
    len(cv_results),
)


for fold, fold_results in cv_results.items():

    print()
    print("-" * 60)
    print(f"Fold {fold}")
    print("-" * 60)

    for model_name in [
        "EEG",
        "VIDEO",
        "FUSION",
        "CONCAT",
        "DECISION",
    ]:

        result = fold_results.get(
            model_name
        )

        if result is None:

            print(
                f"{model_name}: missing"
            )

            continue

        subject_results = result.get(
            "subject_results",
            {},
        )

        print(
            f"{model_name}: "
            f"{len(subject_results)} subjects"
        )

**# 9. CV ANALYSIS & VISUALIZATION**

In [ ]:
# ---------------------------------------------------------
# Build CV result tables
# ---------------------------------------------------------

cv_subject_df = collect_cv_results(
    cv_results
)

cv_fold_df = build_cv_fold_table(
    cv_subject_df
)

cv_summary_df = summarize_cv_results(
    cv_fold_df
)

cv_paper_table = make_cv_paper_table(
    cv_summary_df
)


# ---------------------------------------------------------
# Display main CV tables
# ---------------------------------------------------------

print("Subject-level CV results")
display(cv_subject_df)

print()
print("Fold-level CV results")
display(cv_fold_df)

print()
print("Final CV summary")
display(cv_summary_df)

print()
print("Paper-ready table")
display(cv_paper_table)


# ---------------------------------------------------------
# Model comparisons
# ---------------------------------------------------------

cv_vs_video_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="VIDEO",
)

cv_vs_concat_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="CONCAT",
)

cv_vs_eeg_df = compare_cv_models(
    cv_fold_df,
    metric="best_val_acc",
    baseline="EEG",
)

film_vs_concat_df = compare_two_cv_models(
    cv_fold_df,
    model_a="FUSION",
    model_b="CONCAT",
    metric="best_val_acc",
)

cv_all_metric_comparisons_df = compare_all_metrics(
    cv_fold_df,
    baselines=(
        "VIDEO",
        "CONCAT",
        "EEG",
    ),
)

film_vs_concat_all_metrics_df = (
    compare_film_vs_concat_all_metrics(
        cv_fold_df
    )
)


# ---------------------------------------------------------
# Display model comparisons
# ---------------------------------------------------------

print()
print("Comparison against VIDEO")
display(cv_vs_video_df)

print()
print("Comparison against CONCAT")
display(cv_vs_concat_df)

print()
print("Comparison against EEG")
display(cv_vs_eeg_df)

print()
print("Direct comparison: FiLM vs Concat")
display(film_vs_concat_df)

print()
print("All metric comparisons")
display(cv_all_metric_comparisons_df)

print()
print("FiLM vs Concat across all metrics")
display(film_vs_concat_all_metrics_df)


# ---------------------------------------------------------
# FiLM modulation analysis across CV
# ---------------------------------------------------------

cv_film_stats_df = collect_cv_film_stats(
    cv_results
)

cv_film_summary_df = build_cv_film_summary_table(
    cv_film_stats_df
)


print()
print(
    "FiLM gamma / beta / gate "
    "trial-level analysis"
)
display(cv_film_stats_df)

print()
print(
    "FiLM gamma / beta / gate summary"
)
display(cv_film_summary_df)


# ---------------------------------------------------------
# CV performance plots
# ---------------------------------------------------------

# Final performance with error bars
plot_cv_summary_bar(
    cv_summary_df,
    metric="best_val_acc",
    title=(
        "4-fold cross-validation: "
        "best validation accuracy"
    ),
)


# Stability across folds
plot_cv_fold_lines(
    cv_fold_df,
    metric="best_val_acc",
    title="Model stability across 4 folds",
)


# FiLM vs Concat
plot_cv_model_difference(
    cv_fold_df,
    model_a="FUSION",
    model_b="CONCAT",
    metric="best_val_acc",
)


# FiLM subject heatmap
plot_subject_heatmap(
    cv_subject_df,
    model="FUSION",
    metric="best_val_acc",
    title=(
        "FiLM fusion performance "
        "per subject and fold"
    ),
)


# Concat subject heatmap
plot_subject_heatmap(
    cv_subject_df,
    model="CONCAT",
    metric="best_val_acc",
    title=(
        "Concat fusion performance "
        "per subject and fold"
    ),
)


# ---------------------------------------------------------
# FiLM modulation plots
# ---------------------------------------------------------

plot_film_parameter_histograms(
    cv_film_stats_df
)


# Compare modulation according to class label
plot_film_boxplot_by_label(
    cv_film_stats_df,
    "gamma_deviation_from_1",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "gate_mean",
)

plot_film_boxplot_by_label(
    cv_film_stats_df,
    "relative_modulation",
)


# Compare correct vs incorrect predictions
plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "relative_modulation",
)

plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "beta_abs_mean",
)

plot_film_boxplot_correct_vs_wrong(
    cv_film_stats_df,
    "gate_mean",
)

**# 10. SAVE CV RESULTS**

In [ ]:
# ---------------------------------------------------------
# Output directory
# ---------------------------------------------------------

out_dir = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "cv"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# Core CV tables
# ---------------------------------------------------------

cv_subject_df.to_csv(
    out_dir
    / f"cv4_subject_results_label{LABEL_TYPE}.csv",
    index=False,
)

cv_fold_df.to_csv(
    out_dir
    / f"cv4_fold_results_label{LABEL_TYPE}.csv",
    index=False,
)

cv_summary_df.to_csv(
    out_dir
    / f"cv4_summary_label{LABEL_TYPE}.csv",
    index=False,
)

cv_paper_table.to_csv(
    out_dir
    / f"cv4_paper_table_label{LABEL_TYPE}.csv",
    index=False,
)


# ---------------------------------------------------------
# Model comparison tables
# ---------------------------------------------------------

cv_vs_video_df.to_csv(
    out_dir
    / f"cv4_comparison_vs_video_label{LABEL_TYPE}.csv",
    index=False,
)

cv_vs_concat_df.to_csv(
    out_dir
    / f"cv4_comparison_vs_concat_label{LABEL_TYPE}.csv",
    index=False,
)

cv_vs_eeg_df.to_csv(
    out_dir
    / f"cv4_comparison_vs_eeg_label{LABEL_TYPE}.csv",
    index=False,
)

film_vs_concat_df.to_csv(
    out_dir
    / f"cv4_film_vs_concat_label{LABEL_TYPE}.csv",
    index=False,
)

cv_all_metric_comparisons_df.to_csv(
    out_dir
    / f"cv4_all_metric_comparisons_label{LABEL_TYPE}.csv",
    index=False,
)

film_vs_concat_all_metrics_df.to_csv(
    out_dir
    / f"cv4_film_vs_concat_all_metrics_label{LABEL_TYPE}.csv",
    index=False,
)


# ---------------------------------------------------------
# FiLM analysis tables
# ---------------------------------------------------------

cv_film_stats_df.to_csv(
    out_dir
    / f"cv4_film_stats_label{LABEL_TYPE}.csv",
    index=False,
)

cv_film_summary_df.to_csv(
    out_dir
    / f"cv4_film_summary_label{LABEL_TYPE}.csv",
    index=False,
)


print(
    "Saved CV results to:",
    out_dir,
)